In [1]:
import os
import numpy as np
import pandas as pd
import h5py
import scipy as sp
import scipy.sparse

In [3]:
def decode(x):
    # common pattern in these h5 files
    if isinstance(x, (bytes, np.bytes_)):
        return x.decode("utf-8")
    if isinstance(x, np.ndarray) and x.dtype.kind in ("S", "O"):
        return np.array([decode(i) for i in x])
    return x

def dict_from_group(g):
    out = {}
    for k in g.keys():
        item = g[k]
        if isinstance(item, h5py.Dataset):
            out[k] = item[...]
        elif isinstance(item, h5py.Group):
            out[k] = dict_from_group(item)
    return out

def read_data(filename, sparsify=False, skip_exprs=False):
    with h5py.File(filename, "r") as f:
        obs = pd.DataFrame(dict_from_group(f["obs"]), index=decode(f["obs_names"][...]))
        var = pd.DataFrame(dict_from_group(f["var"]), index=decode(f["var_names"][...]))
        uns = dict_from_group(f["uns"])
        if not skip_exprs:
            exprs_handle = f["exprs"]
            if isinstance(exprs_handle, h5py.Group):
                mat = sp.sparse.csr_matrix(
                    (exprs_handle["data"][...], exprs_handle["indices"][...], exprs_handle["indptr"][...]),
                    shape=exprs_handle["shape"][...]
                )
            else:
                mat = exprs_handle[...].astype(np.float32)
                if sparsify:
                    mat = sp.sparse.csr_matrix(mat)
        else:
            mat = sp.sparse.csr_matrix((obs.shape[0], var.shape[0]))
    return mat, obs, var, uns

def count_nonempty_lines(path):
    n = 0
    with open(path, "r") as f:
        for line in f:
            if line.strip() != "":
                n += 1
    if n == 0:
        raise ValueError(f"No non-empty lines in {path}")
    return n

def subset_text_by_lines(in_path, out_path, keep_idx_sorted_0based):
    """Subset a text file by line number (0-based), streaming (memory-safe)."""
    keep = set(map(int, keep_idx_sorted_0based.tolist()))
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(in_path, "r") as fin, open(out_path, "w") as fout:
        for i, line in enumerate(fin):
            if i in keep:
                fout.write(line)

In [4]:
def _copy_attrs(src_obj, dst_obj):
    for k, v in src_obj.attrs.items():
        try:
            dst_obj.attrs[k] = v
        except Exception:
            pass

def _ensure_group(h5file, group_path):
    if group_path in ("", "/"):
        return h5file
    cur = h5file
    for p in [p for p in group_path.split("/") if p]:
        cur = cur.require_group(p)
    return cur

def _subset_dense_dataset(dset, keep_idx_sorted_0based, n_cells):
    shape = dset.shape
    axes = [ax for ax, s in enumerate(shape) if s == n_cells]
    if not axes:
        return None
    axis = axes[-1]
    slc = [slice(None)] * dset.ndim
    slc[axis] = keep_idx_sorted_0based
    return dset[tuple(slc)]

def _csr_from_group(g):
    return sp.sparse.csr_matrix((g["data"][...], g["indices"][...], g["indptr"][...]), shape=g["shape"][...])

def _write_csr_group(out_group, name, csr):
    gg = out_group.require_group(name)
    # clean old if exists
    for kk in list(gg.keys()):
        del gg[kk]
    gg.create_dataset("data", data=csr.data)
    gg.create_dataset("indices", data=csr.indices)
    gg.create_dataset("indptr", data=csr.indptr)
    gg.create_dataset("shape", data=np.array(csr.shape, dtype=np.int64))

def subset_h5_file(in_h5, out_h5, keep_idx_sorted_0based, n_cells):
    """
    Copy whole H5, but subset datasets aligned with n_cells.
    Special handling for CSR group stored at '/exprs' (common in your read_data()).
    """
    os.makedirs(os.path.dirname(out_h5), exist_ok=True)
    if os.path.exists(out_h5):
        os.remove(out_h5)

    with h5py.File(in_h5, "r") as fin, h5py.File(out_h5, "w") as fout:
        _copy_attrs(fin, fout)

        # If there's an exprs CSR group, handle it explicitly (better than generic per-dataset logic)
        has_exprs_group = ("exprs" in fin) and isinstance(fin["exprs"], h5py.Group) \
                          and all(k in fin["exprs"] for k in ["data", "indices", "indptr", "shape"])

        if has_exprs_group:
            # We'll still do a full copy via visititems, but we will overwrite exprs at the end
            pass

        def visitor(name, obj):
            if isinstance(obj, h5py.Group):
                g_out = _ensure_group(fout, "/" + name)
                _copy_attrs(obj, g_out)
                return

            if not isinstance(obj, h5py.Dataset):
                return

            # Skip datasets under exprs CSR group; we will write them as a unit later
            if has_exprs_group and name.startswith("exprs/"):
                return
            if has_exprs_group and name == "exprs/shape":
                return

            parent_path = os.path.dirname("/" + name)
            dname = os.path.basename(name)
            out_parent = _ensure_group(fout, parent_path)

            sub = None
            try:
                if obj.ndim >= 1 and any(s == n_cells for s in obj.shape):
                    sub = _subset_dense_dataset(obj, keep_idx_sorted_0based, n_cells)
            except Exception:
                sub = None

            if sub is None:
                fin.copy(name, out_parent, name=dname)
                _copy_attrs(obj, out_parent[dname])
            else:
                d_out = out_parent.create_dataset(dname, data=sub, dtype=obj.dtype)
                _copy_attrs(obj, d_out)

        fin.visititems(visitor)

        # Now handle exprs CSR group if present
        if has_exprs_group:
            exprs_csr = _csr_from_group(fin["exprs"])
            exprs_sub = exprs_csr[keep_idx_sorted_0based, :]
            out_root = fout
            # ensure '/exprs' group exists
            _ensure_group(fout, "/exprs")
            _write_csr_group(out_root, "exprs", exprs_sub)
            _copy_attrs(fin["exprs"], fout["exprs"])

In [9]:
base_dir = "/scratch/g/chlin/Yushu/Data/Plasschaert"

in_h5   = os.path.join(base_dir, "data.h5")
in_norm = os.path.join(base_dir, "data_norm.txt")
in_ct   = os.path.join(base_dir, "data_celltype.txt")

assert os.path.exists(in_h5),   f"Missing {in_h5}"
assert os.path.exists(in_norm), f"Missing {in_norm}"
assert os.path.exists(in_ct),   f"Missing {in_ct}"

# Determine n_cells from celltype file (one line per cell)
n_cells = count_nonempty_lines(in_ct)
print("n_cells =", n_cells)

seed = 123
rng = np.random.default_rng(seed)

# One permutation -> nested subsets (p40 ⊂ p60 ⊂ p80)
perm = rng.permutation(n_cells)

ratios = [0.4, 0.6, 0.8]
for r in ratios:
    k = int(np.floor(r * n_cells))
    keep_idx = np.sort(perm[:k])  # sorted so text order remains original

    tag = f"p{int(round(r*100))}"
    out_dir = os.path.join(base_dir, tag)
    os.makedirs(out_dir, exist_ok=True)

    # Save indices
    np.savetxt(os.path.join(out_dir, "selected_idx_0based.txt"), keep_idx, fmt="%d")

    # Subset txt files by line
    subset_text_by_lines(in_norm, os.path.join(out_dir, "data_norm.txt"), keep_idx)
    subset_text_by_lines(in_ct,   os.path.join(out_dir, "data_celltype.txt"), keep_idx)

    # Subset H5
    subset_h5_file(in_h5, os.path.join(out_dir, "data.h5"), keep_idx, n_cells)

    print(f"[OK] {tag}: kept {k}/{n_cells} -> {out_dir}")

n_cells = 6977
[OK] p40: kept 2790/6977 -> /scratch/g/chlin/Yushu/Data/Plasschaert/p40
[OK] p60: kept 4186/6977 -> /scratch/g/chlin/Yushu/Data/Plasschaert/p60
[OK] p80: kept 5581/6977 -> /scratch/g/chlin/Yushu/Data/Plasschaert/p80


In [10]:
# Check txt sizes
for tag in ["p40", "p60", "p80"]:
    ct_path = os.path.join(base_dir, tag, "data_celltype.txt")
    nn = count_nonempty_lines(ct_path)
    print(tag, "celltype lines =", nn)

# Check H5 basic consistency
for tag in ["p40", "p60", "p80"]:
    h5p = os.path.join(base_dir, tag, "data.h5")
    with h5py.File(h5p, "r") as f:
        keys = list(f.keys())
        print(tag, "top-level keys:", keys[:10], "...")
        if "Y" in f:
            print("  Y shape:", f["Y"].shape)
        if "obs_names" in f:
            print("  obs_names shape:", f["obs_names"].shape)
        if "exprs" in f and isinstance(f["exprs"], h5py.Group) and "shape" in f["exprs"]:
            print("  exprs shape:", f["exprs"]["shape"][...])
        elif "exprs" in f and isinstance(f["exprs"], h5py.Dataset):
            print("  exprs shape:", f["exprs"].shape)

p40 celltype lines = 2790
p60 celltype lines = 4186
p80 celltype lines = 5581
p40 top-level keys: ['exprs', 'obs', 'obs_names', 'uns', 'var', 'var_names'] ...
  obs_names shape: (2790,)
  exprs shape: [ 2790 28205]
p60 top-level keys: ['exprs', 'obs', 'obs_names', 'uns', 'var', 'var_names'] ...
  obs_names shape: (4186,)
  exprs shape: [ 4186 28205]
p80 top-level keys: ['exprs', 'obs', 'obs_names', 'uns', 'var', 'var_names'] ...
  obs_names shape: (5581,)
  exprs shape: [ 5581 28205]


In [7]:
import sys, scipy
print("python:", sys.executable)
print("scipy:", scipy.__version__, scipy.__file__)

try:
    import scipy.sparse as sps
    print("✅ scipy.sparse import OK:", sps.__file__)
except Exception as e:
    print("❌ scipy.sparse import FAILED:", repr(e))

python: /scratch/g/chlin/Yushu/envs/scdac/bin/python
scipy: 1.7.3 /scratch/g/chlin/Yushu/envs/scdac/lib/python3.7/site-packages/scipy/__init__.py
✅ scipy.sparse import OK: /scratch/g/chlin/Yushu/envs/scdac/lib/python3.7/site-packages/scipy/sparse/__init__.py


## scgnn

In [1]:
import os
import numpy as np
import pandas as pd

base_dir = "/scratch/g/chlin/Yushu/scGNN/Data/Plasschaert"
in_counts = os.path.join(base_dir, "counts.csv")
idx_base = "/scratch/g/chlin/Yushu/Data/Plasschaert/"

assert os.path.exists(in_counts), f"Missing {in_counts}"

# Read counts.csv
# If counts.csv has a header row (common), keep header=0; if not, set header=None
df = pd.read_csv(in_counts, header=0)

print("counts.csv shape:", df.shape)

for tag in ["p40", "p60", "p80"]:
    idx_path = os.path.join(idx_base, tag, "selected_idx_0based.txt")
    assert os.path.exists(idx_path), f"Missing {idx_path}"

    idx = np.loadtxt(idx_path, dtype=int)
    idx = np.atleast_1d(idx)

    out_dir = os.path.join(base_dir, tag)
    os.makedirs(out_dir, exist_ok=True)

    out_counts = os.path.join(out_dir, "counts.csv")

    # subset rows by same indices (0-based)
    df_sub = df.iloc[idx, :]

    df_sub.to_csv(out_counts, index=False)
    print(f"[OK] {tag}: wrote {df_sub.shape} -> {out_counts}")

counts.csv shape: (28205, 6978)
[OK] p40: wrote (2790, 6978) -> /scratch/g/chlin/Yushu/scGNN/Data/Plasschaert/p40/counts.csv
[OK] p60: wrote (4186, 6978) -> /scratch/g/chlin/Yushu/scGNN/Data/Plasschaert/p60/counts.csv
[OK] p80: wrote (5581, 6978) -> /scratch/g/chlin/Yushu/scGNN/Data/Plasschaert/p80/counts.csv


## scdac

In [3]:
import os
import numpy as np
import pandas as pd
import shutil
from pathlib import Path

# where your saved indices live (from earlier p40/p60/p80 step)
idx_base = "/scratch/g/chlin/Yushu/Data/Plasschaert/"  # contains p40/p60/p80/selected_idx_0based.txt

# scDAC inputs
vec_in_dir = "/scratch/g/chlin/Yushu/scDAC/scDAC/data/Plasschaert/subset_0/vec/rna"
label_in   = "/scratch/g/chlin/Yushu/scDAC/scDAC/data/Plasschaert/label.csv"

assert os.path.isdir(vec_in_dir), f"Missing dir: {vec_in_dir}"
assert os.path.exists(label_in), f"Missing: {label_in}"

tags = ["p40", "p60", "p80"]

# load indices (0-based)
idx_map = {}
for tag in tags:
    idx_path = os.path.join(idx_base, tag, "selected_idx_0based.txt")
    assert os.path.exists(idx_path), f"Missing: {idx_path}"
    idx = np.loadtxt(idx_path, dtype=int)
    idx = np.atleast_1d(idx)
    idx_map[tag] = np.sort(idx)

print({k: len(v) for k,v in idx_map.items()})

{'p40': 2790, 'p60': 4186, 'p80': 5581}


In [4]:
def detect_vec_filename_pattern(vec_dir, sample_idx_0based, pad_options=(3,4,5,6)):
    """
    Try patterns like:
      {i:0{pad}d}.csv where i can be idx or idx+1
    and choose the (offset,pad) with best hit rate over sample indices.
    """
    vec_dir = Path(vec_dir)
    best = None
    best_hits = -1
    sample = np.asarray(sample_idx_0based, dtype=int)
    # use up to 200 samples for speed
    if len(sample) > 200:
        sample = sample[:200]

    for offset in [0, 1]:  # 0-based files or 1-based files
        for pad in pad_options:
            hits = 0
            for idx0 in sample:
                fname = f"{idx0 + offset:0{pad}d}.csv"
                if (vec_dir / fname).exists():
                    hits += 1
            if hits > best_hits:
                best_hits = hits
                best = (offset, pad)

    return best, best_hits, len(sample)

best, hits, ntest = detect_vec_filename_pattern(vec_in_dir, idx_map["p80"])
offset, pad = best
print(f"Detected pattern: offset={offset} (file index = idx+offset), pad={pad}, hit_rate={hits}/{ntest}")

Detected pattern: offset=0 (file index = idx+offset), pad=4, hit_rate=200/200


In [7]:
def copy_selected_vec_files(vec_dir_in, vec_dir_out, keep_idx_0based, offset, pad):
    vec_dir_in = Path(vec_dir_in)
    vec_dir_out = Path(vec_dir_out)
    vec_dir_out.mkdir(parents=True, exist_ok=True)

    missing = []
    copied = 0

    for idx0 in keep_idx_0based:
        fname = f"{idx0 + offset:0{pad}d}.csv"
        src = vec_dir_in / fname
        dst = vec_dir_out / fname
        if src.exists():
            shutil.copy2(src, dst)
            copied += 1
        else:
            missing.append(fname)

    return copied, missing

# output base for scDAC subsets (you can rename these folders if you prefer)
out_base = "/scratch/g/chlin/Yushu/scDAC/scDAC/data/Plasschaert_downsample"
#os.makedirs(out_base, exist_ok=True)

for tag in tags:
    out_vec = os.path.join(out_base, tag, "vec", "rna")
    keep = idx_map[tag]
    copied, missing = copy_selected_vec_files(vec_in_dir, out_vec, keep, offset=offset, pad=pad)
    print(f"[OK] {tag}: copied {copied}/{len(keep)} vec files -> {out_vec}")
    if missing:
        print(f"  Missing {len(missing)} files (showing up to 10): {missing[:10]}")

[OK] p40: copied 2790/2790 vec files -> /scratch/g/chlin/Yushu/scDAC/scDAC/data/Plasschaert_downsample/p40/vec/rna
[OK] p60: copied 4186/4186 vec files -> /scratch/g/chlin/Yushu/scDAC/scDAC/data/Plasschaert_downsample/p60/vec/rna
[OK] p80: copied 5581/5581 vec files -> /scratch/g/chlin/Yushu/scDAC/scDAC/data/Plasschaert_downsample/p80/vec/rna


In [8]:
# read label.csv
label_df = pd.read_csv(label_in)
print("label.csv shape:", label_df.shape)

for tag in tags:
    keep = idx_map[tag]
    out_dir = os.path.join(out_base, tag)
    os.makedirs(out_dir, exist_ok=True)

    out_label = os.path.join(out_dir, "label.csv")

    # subset rows by index
    label_sub = label_df.iloc[keep, :].reset_index(drop=True)
    label_sub.to_csv(out_label, index=False)

    print(f"[OK] {tag}: label subset shape {label_sub.shape} -> {out_label}")

label.csv shape: (6977, 2)
[OK] p40: label subset shape (2790, 2) -> /scratch/g/chlin/Yushu/scDAC/scDAC/data/Plasschaert_downsample/p40/label.csv
[OK] p60: label subset shape (4186, 2) -> /scratch/g/chlin/Yushu/scDAC/scDAC/data/Plasschaert_downsample/p60/label.csv
[OK] p80: label subset shape (5581, 2) -> /scratch/g/chlin/Yushu/scDAC/scDAC/data/Plasschaert_downsample/p80/label.csv


In [9]:
for tag in tags:
    out_vec = Path(out_base) / tag / "vec" / "rna"
    n_files = len(list(out_vec.glob("*.csv")))
    out_label = Path(out_base) / tag / "label.csv"
    print(tag, "vec files:", n_files, "| label exists:", out_label.exists())

p40 vec files: 2790 | label exists: True
p60 vec files: 4186 | label exists: True
p80 vec files: 5581 | label exists: True
